# RQ2 TIB: sample-size EDA code (results withdrawn)

The prior 20/70 Lecture-cohort score analysis and its saved outputs were withdrawn on 2026-10-07. Code is retained for a future reanalysis. Current dataset scope is all pinned train, valid, and test records without a genre filter.


In [ ]:
import hashlib, json, math, statistics
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def find_repo(start):
    for parent in (start, *start.parents):
        if (parent / 'experiments/scripts/rq2-tib-valid-eda.py').exists():
            return parent
    raise FileNotFoundError('Run from inside the nested multimodal-lecture-summarizer repository')

ROOT = find_repo(Path.cwd().resolve())
RECEIPT = ROOT / 'plans/261002-0946-rq2-controlled-summarization/rq2-tib-valid-diagnostic-receipt.json'
ELIGIBLE = ROOT / 'plans/261002-0946-rq2-controlled-summarization/rq2-tib-valid-eligible-metadata.json'
EVALUATION = ROOT / 'plans/261002-0946-rq2-controlled-summarization/rq2-v2-automatic-evaluation.json'
receipt = json.loads(RECEIPT.read_text(encoding='utf-8'))
eligible_snapshot = json.loads(ELIGIBLE.read_text(encoding='utf-8'))
evaluation_bytes = EVALUATION.read_bytes()
assert hashlib.sha256(evaluation_bytes).hexdigest() == 'd65bf94ce198c52a093433bd62583542e3b96476f9587afb63ba843ce78552f9'
evaluation = json.loads(evaluation_bytes)
assert receipt['revision'] == evaluation['revision']
assert receipt['selected_doi_sha256'] == evaluation['selection_sha256']
assert eligible_snapshot['dataset'] == receipt['dataset']
assert eligible_snapshot['revision'] == receipt['revision']
assert eligible_snapshot['split'] == 'valid' and not eligible_snapshot['test_content_accessed']
assert eligible_snapshot['selection_sha256'] == receipt['selected_doi_sha256']
selected = pd.DataFrame(receipt['selected'])
eligible = pd.DataFrame(eligible_snapshot['records'])
scores = pd.DataFrame(evaluation['rows'])
assert len(selected) == len(scores) == 20
assert len(eligible) == eligible_snapshot['eligible_count'] == receipt['candidate_count'] == 70
assert eligible.doi.is_unique and set(selected.doi) <= set(eligible.doi)
assert set(selected.doi) == set(scores.doi)
assert (scores.status == 'completed').all()
eligible['author_key'] = eligible.author.map(lambda value: ' '.join(str(value).casefold().split()))
assert eligible.author_key.ne('').all()
eligible['doi_hash'] = eligible.doi.map(lambda value: hashlib.sha256(value.encode()).hexdigest())
one_per_author = eligible.sort_values(['doi_hash', 'doi']).drop_duplicates('author_key').copy()
assert len(one_per_author) == 52
assert set(selected.doi) <= set(one_per_author.doi)
print('Revision:', receipt['revision'], '| split:', receipt['split_read'])
print('All valid:', receipt['source_count'], '| exact Lecture:', receipt['lecture_count'],
      '| structurally eligible:', len(eligible), '| unique authors:', len(one_per_author),
      '| selected:', len(selected))
print('Selection:', receipt['selection'])
print('V2 evaluation SHA-256:', hashlib.sha256(evaluation_bytes).hexdigest())

## What the 20 cases cover

The original selector chose the first 20 eligible DOI hashes with one normalized author per item. It did not stratify by `subject`, length, or abstract quality. The executed export now includes metadata for all 70 structurally eligible talks. A deterministic one-per-author extension contains 52 talks and retains the original 20. This is a candidate cohort, not a frozen formal endpoint. The existing abstract screen is a provisional editorial screen, not verified ground truth.

In [ ]:
SCREEN = ROOT / 'plans/261002-0946-rq2-controlled-summarization/rq2-tib-valid-reference-screen.json'
screen = json.loads(SCREEN.read_text(encoding='utf-8'))
assert screen['revision'] == receipt['revision'] and screen['split'] == 'valid'
selected['doi_id'] = selected.doi.str.extract(r'/(\d+)')[0]
selected['reference_screen'] = selected.doi_id.map(lambda d: screen['items'][d]['screen'])
subjects = pd.concat([eligible.subject.value_counts().rename('eligible_70'),
                      one_per_author.subject.value_counts().rename('one_author_52'),
                      selected.subject.value_counts().rename('selected_20')], axis=1).fillna(0).astype(int)
subjects.index.name = 'subject'
subjects['eligible_pct'] = 100 * subjects.eligible_70 / len(eligible)
subjects['selected_pct'] = 100 * subjects.selected_20 / len(selected)
print('Subject distribution: eligible vs author-distinct extension vs original 20')
display(subjects.round(1))
print('Subject × provisional abstract screen:')
display(pd.crosstab(selected.subject, selected.reference_screen))
print('Abstract screen totals:', selected.reference_screen.value_counts().to_dict())
for name, frame in [('eligible_70', eligible), ('one_author_52', one_per_author), ('selected_20', selected)]:
    print(name, 'short abstracts', int((frame.abstract_words < 30).sum()), '/', len(frame),
          '| words median', int(frame.transcript_words.median()),
          '| words max', int(frame.transcript_words.max()),
          '| duration median min', round(frame.duration_minutes.median(), 1))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
subjects[['eligible_pct', 'selected_pct']].plot.barh(ax=axes[0], title='Subject share (%)')
eligible.transcript_words.hist(ax=axes[1], bins=10, alpha=.55, label='eligible 70')
selected.transcript_words.hist(ax=axes[1], bins=10, alpha=.65, label='selected 20')
axes[1].legend(); axes[1].set(title='Transcript length', xlabel='Words')
fig.tight_layout(); plt.show()

## Paired-score precision analysis — withdrawn

This code used a removed diagnostic score file. Its old estimates are not current evidence. Re-run only after a new qualified cohort and scoring protocol are fixed.


In [ ]:
delta = scores.delta_S3_minus_S1.astype(float)
sd = statistics.stdev(delta)
mean = statistics.mean(delta)
ci = evaluation['paired_rouge']['ci95']
print('Observed n:', len(delta), '| mean delta:', round(mean, 5), '| SD:', round(sd, 5))
print('Saved paired-bootstrap 95% CI:', [round(x, 5) for x in ci])
print('S3 / S1 / tie:', int((delta > 0).sum()), '/', int((delta < 0).sum()), '/', int((delta == 0).sum()))
scenarios = pd.DataFrame([{'n': n, 'approx_95pct_half_width': 1.96 * sd / math.sqrt(n)}
                          for n in [20, 30, 40, 50, 52, 60, 70]])
display(scenarios.round(5))
for width in [0.01, 0.005]:
    needed = math.ceil((1.96 * sd / width) ** 2)
    print(f'Target ±{width:.3f} ROUGE-L -> about {needed} independent pairs under v2 variance')
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
delta.hist(ax=axes[0], bins=8); axes[0].axvline(0, color='black', linestyle='--')
axes[0].set(title='20 paired ROUGE-L deltas', xlabel='S3 − S1')
scenarios.plot(x='n', y='approx_95pct_half_width', marker='o', ax=axes[1], legend=False)
axes[1].axhline(0.01, color='darkred', linestyle='--')
axes[1].set(title='Illustrative precision', ylabel='Approx. 95% half-width')
fig.tight_layout(); plt.show()

## Current decision

Do not use the former paired-score estimates or sample-size recommendation. Complete all-split TIB EDA, qualify references and groups, then design a new evaluation and precision analysis.
